In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from load_dataset import *
from prepare_datasets import *
from tokenizer import *
from utils import *

In [3]:
load_dataset()

Скачивание SlavaYus/tinysms_qa_12m в /Users/artyom-step/nlp-hw2/data/tinysms_qa_12m


Fetching 64 files:   0%|          | 0/64 [00:00<?, ?it/s]

Датасет сохранён в /Users/artyom-step/nlp-hw2/data/tinysms_qa_12m


In [4]:
dataset = load_dataset_from_files(DEFAULT_OUTPUT_DIR)

Resolving data files:   0%|          | 0/60 [00:00<?, ?it/s]

In [5]:
dataset = remove_dataset_comments(dataset)
train, test = dataset["train"], dataset["test"]
train_datasets = split_dataset(
    dataset["train"],
    sizes={"train": 0.8, "val": 0.1, "test": 0.1},
    seed=42,
    columns=["question", "code"],
)

In [6]:
tokenizer = train_bpe_tokenizer(train_datasets, vocab_size=8000)

In [7]:
from models import BaselineTransformer, TransformerConfig
from utils import get_device

device = get_device()
config = TransformerConfig.from_tokenizer(tokenizer)
model = BaselineTransformer(config).to(device)

In [ ]:
from training import TrainingConfig, train_model

config = TrainingConfig(
    tokenizer=tokenizer,
    output_dir="artifacts/experiments/baseline_001",
    
)
result = train_model(train_datasets, model, config)

Обучение: 9,463,735 строк; валидация: 1,182,887


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
from inference import InferenceConfig, generate_code
from evaluation import evaluate_code
from plotting import plot_evaluation

generation = generate_code(
    train_datasets["val"], model, InferenceConfig(tokenizer=tokenizer)
)
evaluation = evaluate_code(generation)
print(evaluation.metrics)
plot_evaluation(evaluation)